In [21]:
import dash
import plotly.express as px
import plotly.graph_objects as go
import dash_bootstrap_components as dbc
import pandas as pd
from dash import html, dcc, Input, Output

df = pd.read_csv('./clinical_analytics.csv')

df['Wait Time Min'] = pd.to_numeric(df['Wait Time Min'], errors='coerce')
df['Care Score'] = pd.to_numeric(df['Care Score'], errors='coerce')
df = df.dropna(subset=['Wait Time Min', 'Care Score'])

negative_count = (df['Wait Time Min'] < 0).sum()
df = df[df['Wait Time Min'] >= 0]

app = dash.Dash(external_stylesheets=[dbc.themes.MATERIA])

app.layout = dbc.Container([
    dbc.NavbarSimple(
        brand="Dashboard de Analítica Clínica",
        color="primary",
        dark=True,
        className="mb-4"
    ),
    
    dbc.Alert(
        f"Nota: Se encontraron {negative_count} registros con tiempos de espera negativos (pacientes que llegaron tarde) y fueron excluidos del análisis.",
        color="warning",
        className="mb-3"
    ) if negative_count > 0 else html.Div(),
    
    dbc.Row([
        dbc.Col(dbc.Card([
            dbc.CardBody([
                html.H5("Tiempo de Espera Promedio", className="card-title"),
                html.H2(id='avg-wait', className="text-success")
            ])
        ]), md=4),
        dbc.Col(dbc.Card([
            dbc.CardBody([
                html.H5("Care Score Promedio", className="card-title"),
                html.H2(id='avg-care-score', className="text-info")
            ])
        ]), md=4),
        dbc.Col(dbc.Card([
            dbc.CardBody([
                html.H5("Total de Registros", className="card-title"),
                html.H2(id='total-records', className="text-warning")
            ])
        ]), md=4),
    ], className="mb-4"),
    
    dbc.Row([
        dbc.Col([
            html.Label("Seleccionar Departamento:", className="fw-bold"),
            dcc.Dropdown(
                id="department-filter",
                options=[{'label': dept, 'value': dept} for dept in sorted(df['Department'].unique())],
                value=df['Department'].unique().tolist(),
                multi=True,
                className="mb-3"
            )
        ], md=4),
        
        dbc.Col([
            html.Label("Tipo de Admisión:", className="fw-bold"),
            dcc.RadioItems(
                id="admit-type-filter",
                options=[{'label': atype, 'value': atype} for atype in df['Admit Type'].unique()],
                value=df['Admit Type'].unique()[0],
                inline=False,
                labelClassName="d-block"
            )
        ], md=4),
        
        dbc.Col([
            html.Label("Rango de Tiempo de Espera (minutos):", className="fw-bold"),
            dcc.RangeSlider(
                id="wait-time-filter",
                min=df['Wait Time Min'].min(),
                max=df['Wait Time Min'].max(),
                step=1,
                value=[df['Wait Time Min'].min(), df['Wait Time Min'].max()],
                marks={
                    int(df['Wait Time Min'].min()): f"{int(df['Wait Time Min'].min())}",
                    int(df['Wait Time Min'].max()//2): f"{int(df['Wait Time Min'].max()//2)}",
                    int(df['Wait Time Min'].max()): f"{int(df['Wait Time Min'].max())}"
                },
                tooltip={"placement": "bottom", "always_visible": True}
            )
        ], md=4)
    ], className="mb-4"),
    
    dbc.Row([
        dbc.Col(dcc.Graph(id="scatter-plot"), md=6),
        dbc.Col(dcc.Graph(id="pie-plot"), md=6)
    ], className="mb-4"),
    
    dbc.Row([
        dbc.Col(dcc.Graph(id="histogram-plot"), md=12)
    ], className="mb-4")
], fluid=True)

@app.callback(
    [
        Output("scatter-plot", "figure"),
        Output("pie-plot", "figure"),
        Output("histogram-plot", "figure"),
        Output("avg-wait", "children"),
        Output("avg-care-score", "children"),
        Output("total-records", "children"),
    ],
    [
        Input("department-filter", "value"),
        Input("admit-type-filter", "value"),
        Input("wait-time-filter", "value")
    ]
)
def update_dashboard(departments, admit_type, wait_range):
    dff = df[
        (df['Department'].isin(departments)) & 
        (df['Admit Type'] == admit_type) & 
        (df['Wait Time Min'].between(wait_range[0], wait_range[1]))
    ]
    
    scatter = px.scatter(
        dff, 
        x="Wait Time Min", 
        y="Care Score",
        color="Department",
        size="Care Score",
        hover_data=['Clinic Name', 'Encounter Status'],
        title="Tiempo de Espera vs. Care Score por Departamento",
        labels={
            "Wait Time Min": "Tiempo de Espera (min)",
            "Care Score": "Puntuación de Atención"
        }
    )
    scatter.update_layout(height=400)
    
    dept_counts = dff['Department'].value_counts().reset_index()
    dept_counts.columns = ['Department', 'Count']
    
    pie = px.pie(
        dept_counts,
        names="Department",
        values="Count",
        title="Distribución de Casos por Departamento",
        hole=0.3
    )
    pie.update_layout(height=400)
    
    histogram = px.histogram(
        dff,
        x="Wait Time Min",
        nbins=30,
        color="Encounter Status",
        title="Distribución de Tiempos de Espera",
        labels={"Wait Time Min": "Tiempo de Espera (min)", "count": "Frecuencia"},
        marginal="box"
    )
    histogram.update_layout(height=400)
    
    avg_wait = f"{dff['Wait Time Min'].mean():.1f} min" if not dff.empty else "No hay datos"
    avg_care = f"{dff['Care Score'].mean():.2f}" if not dff.empty else "No hay datos"
    total_records = f"{len(dff):,}" if not dff.empty else "0"
    
    return scatter, pie, histogram, avg_wait, avg_care, total_records


@app.callback(
    Output("point-info", "children"),
    Input("scatter-plot", "clickData")
)
def display_click_data(clickData):
    
    point = clickData["points"][0]
    
    return dbc.Card([
        dbc.CardHeader("Detalles del Punto Seleccionado", className="fw-bold"),
        dbc.CardBody([
            html.P([
                html.Strong("Tiempo de Espera: "), 
                f"{point['x']:.1f} minutos"
            ]),
            html.P([
                html.Strong("Care Score: "), 
                f"{point['y']}"
            ]),
            html.P([
                html.Strong("Departamento: "), 
                point.get('customdata', ['N/A'])[0] if 'customdata' in point else 'N/A'
            ]),
        ])
    ], className="shadow-sm")


if __name__ == '__main__':
    app.run(debug=True, port=8005)